## Importing libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk

## Importing Dataset

In [2]:
from datasets import load_dataset
wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.en", streaming=True, split="train")
print("Wiki Dataset")
print(wiki_dataset)

/home/saba_2422cs15/learn_pytorch/pytorch_from_scratch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wiki Dataset
IterableDataset({
    features: ['id', 'url', 'title', 'text'],
    num_shards: 41
})


In [75]:
type(wiki_dataset)

datasets.iterable_dataset.IterableDataset

In [3]:
wiki_dataset.features

{'id': Value('string'),
 'url': Value('string'),
 'title': Value('string'),
 'text': Value('string')}

In [4]:
for example in wiki_dataset:
    document = example['text']
    break

In [5]:
print(document[:500])

Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as


## Tokenization

In [6]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/saba_2422cs15/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/saba_2422cs15/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
tokens = word_tokenize(document.lower()) # There will also be repeated tokens

## Building Dictionary

In [8]:
# dictionary = {'<unk>':0}
dictionary = {'<pad>': 0, '<unk>': 1}
# Counter(tokens).keys()
# for token in Counter(tokens).keys():
#     print(token)

In [9]:
for token in Counter(tokens).keys():
    if token not in dictionary:
        dictionary[token] = len(dictionary)

In [10]:
for key, value in list(dictionary.items())[:10]:
    print(key, value)

<pad> 0
<unk> 1
anarchism 2
is 3
a 4
political 5
philosophy 6
and 7
movement 8
that 9


In [11]:
len(dictionary)

1992

## Extracting sentences from data

In [12]:
for i in range(3):
    print(f"{i}th line")
    print(document.split('\n')[i])

0th line
Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).
1th line

2th line
Humans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During the latter half of the 19th and the first decades of the 20th centur

In [13]:
import nltk
from nltk.tokenize import sent_tokenize
sentence = sent_tokenize(document)

In [14]:
import re

def clean_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z\s]', '', sentence) # Keep only letters and spaces
    sentence = re.sub(r'\s+', ' ', sentence).strip() # Remove extra whitespace
    return sentence

cleaned_sentences = [clean_sentence(s) for s in sentence]
input_sentences = [s for s in cleaned_sentences if len(s.split()) > 2] # Filter short sentences

## Assigning index numbers to words for the sentences

In [15]:
def text_to_indices(sentence, dictionary):
    numerical_sentence = []
    for word in sentence:
        if word in dictionary:
            numerical_sentence.append(dictionary[word])
        else:
            numerical_sentence.append(dictionary['<unk>'])
    return numerical_sentence

In [16]:
input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), dictionary))

In [17]:
type(input_numerical_sentences)

list

In [18]:
input_numerical_sentences[0]

[2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 3,
 10,
 11,
 12,
 13,
 14,
 15,
 7,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 7,
 26,
 28,
 29,
 1,
 7,
 31]

In [19]:
len(input_numerical_sentences)

294

## Forming sequences

In [20]:
for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
        print(sentence[:i+1])
    break

[2, 3]
[2, 3, 4]
[2, 3, 4, 5]
[2, 3, 4, 5, 6]
[2, 3, 4, 5, 6, 7]
[2, 3, 4, 5, 6, 7, 8]
[2, 3, 4, 5, 6, 7, 8, 9]
[2, 3, 4, 5, 6, 7, 8, 9, 3]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17, 18]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17, 18, 19]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17, 18, 19, 20]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17, 18, 19, 20, 21]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 16, 17, 18, 19, 20, 21, 22]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 12, 13, 14, 15, 7, 1

In [21]:
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
        # print(sentence[:i+1])
        training_sequence.append(sentence[:i+1])

In [22]:
for i in range(10):
    print(training_sequence[i])

[2, 3]
[2, 3, 4]
[2, 3, 4, 5]
[2, 3, 4, 5, 6]
[2, 3, 4, 5, 6, 7]
[2, 3, 4, 5, 6, 7, 8]
[2, 3, 4, 5, 6, 7, 8, 9]
[2, 3, 4, 5, 6, 7, 8, 9, 3]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10]
[2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11]


In [23]:
len(training_sequence)

6396

In [24]:
training_sequence[:6]

[[2, 3],
 [2, 3, 4],
 [2, 3, 4, 5],
 [2, 3, 4, 5, 6],
 [2, 3, 4, 5, 6, 7],
 [2, 3, 4, 5, 6, 7, 8]]

## Zero Padding

In [25]:
len_list = []
for sequence in training_sequence:
    len_list.append(len(sequence))
print(max(len_list))

57


In [26]:
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append([0]*(max(len_list)-len(sequence))+sequence)
len(padded_training_sequence)

6396

In [27]:
print(len(padded_training_sequence[0]))
print(len(padded_training_sequence[1]))
print(len(padded_training_sequence[2]))

57
57
57


In [28]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        [  0,   0,   0,  ...,   3,   4,   5],
        ...,
        [  0,   0,   0,  ...,   5, 150, 861],
        [  0,   0,   0,  ..., 150, 861, 998],
        [  0,   0,   0,  ..., 861, 998,  58]])

In [29]:
padded_training_sequence.shape

torch.Size([6396, 57])

In [30]:
x = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [31]:
x

tensor([[   0,    0,    0,  ...,    0,    0,    2],
        [   0,    0,    0,  ...,    0,    2,    3],
        [   0,    0,    0,  ...,    2,    3,    4],
        ...,
        [   0,    0,    0,  ..., 1476,    5,  150],
        [   0,    0,    0,  ...,    5,  150,  861],
        [   0,    0,    0,  ...,  150,  861,  998]])

In [32]:
y

tensor([  3,   4,   5,  ..., 861, 998,  58])

In [33]:
class CustomDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self,index):
        return self.x[index],self.y[index]

In [34]:
dataset = CustomDataset(x,y)

In [35]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 2]),
 tensor(3))

In [36]:
len(dataset)

6396

In [37]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [38]:
# for x, y in dataloader:
#     print(x,y)

In [39]:
class LSTMModel(nn.Module):
    def __init__(self, dictionary_size):
        super().__init__()
        self.embedding = nn.Embedding(dictionary_size,100)
        self.lstm = nn.LSTM(100, 150, batch_first = True)
        self.fc = nn.Linear(150,dictionary_size)
    def forward(self,x):
        embedded = self.embedding(x)
        intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))
        return output

## For Clear Picture Starts

In [40]:
# x = nn.Embedding(289,100)
# y = nn.LSTM(100,150,batch_first = True)

In [41]:
# dataset[0][0].shape

In [42]:
# dataset[0][0].unsqueeze(0).shape

In [43]:
# a = dataset[0][0].unsqueeze(0)
# a

In [44]:
# x(a)

In [45]:
# x(a).shape

In [46]:
# b = x(a)

In [47]:
# y(b)

In [48]:
# len(y(b))

In [49]:
# c,d = y(b)

In [50]:
# print(c.shape) # C is the set of all intermediate states.
# print(len(d))

In [51]:
# e,f = d

In [52]:
# print(e.shape) # Findal hidden state
# print(f.shape) # Final cell state

In [53]:
# z = nn.Linear(150,289)

In [54]:
# z(f.squeeze(0))

In [55]:
# z(f.squeeze(0)).shape

## For Clear Picture Ends

In [56]:
model = LSTMModel(len(dictionary))

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [58]:
model.to(device)

LSTMModel(
  (embedding): Embedding(1992, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=1992, bias=True)
)

In [59]:
epochs = 50
learning_rate = 0.001

In [60]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = learning_rate)

## Training Loop

In [61]:
for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()
    avg_loss = total_loss/len(dataloader)
    print(f"Epoch: {epoch + 1}, Loss: {avg_loss:.4f}")

Epoch: 1, Loss: 6.6550
Epoch: 2, Loss: 5.9488
Epoch: 3, Loss: 5.4876
Epoch: 4, Loss: 4.9755
Epoch: 5, Loss: 4.4490
Epoch: 6, Loss: 3.9319
Epoch: 7, Loss: 3.4262
Epoch: 8, Loss: 2.9437
Epoch: 9, Loss: 2.5002
Epoch: 10, Loss: 2.1015
Epoch: 11, Loss: 1.7491
Epoch: 12, Loss: 1.4495
Epoch: 13, Loss: 1.1990
Epoch: 14, Loss: 0.9941
Epoch: 15, Loss: 0.8207
Epoch: 16, Loss: 0.6797
Epoch: 17, Loss: 0.5683
Epoch: 18, Loss: 0.4739
Epoch: 19, Loss: 0.4020
Epoch: 20, Loss: 0.3437
Epoch: 21, Loss: 0.2984
Epoch: 22, Loss: 0.2615
Epoch: 23, Loss: 0.2335
Epoch: 24, Loss: 0.2092
Epoch: 25, Loss: 0.1928
Epoch: 26, Loss: 0.1786
Epoch: 27, Loss: 0.1657
Epoch: 28, Loss: 0.1550
Epoch: 29, Loss: 0.1493
Epoch: 30, Loss: 0.1408
Epoch: 31, Loss: 0.1340
Epoch: 32, Loss: 0.1287
Epoch: 33, Loss: 0.1259
Epoch: 34, Loss: 0.1221
Epoch: 35, Loss: 0.1185
Epoch: 36, Loss: 0.1160
Epoch: 37, Loss: 0.1141
Epoch: 38, Loss: 0.1108
Epoch: 39, Loss: 0.1091
Epoch: 40, Loss: 0.1074
Epoch: 41, Loss: 0.1053
Epoch: 42, Loss: 0.1044
E

## Prediction

In [62]:
def prediction(model,dictionary,text):
    tokenized_text = nltk.word_tokenize(text.lower()) # tokenize
    numerical_text = text_to_indices(tokenized_text,dictionary) # Text to numerical indices
    padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text,dtype = torch.long).unsqueeze(0).to(device) # padding
    output = model(padded_text) # send to model
    value, index = torch.max(output, dim=1) # predicted index
    return text + " " + list(dictionary.keys())[index] # Merge with text

In [69]:
prediction(model,dictionary,"Humans have lived")

'Humans have lived in'

In [70]:
import time
def show_prediction(input_text,num_words):
    for i in range(num_words):
        output_text = prediction(model,dictionary,input_text)
        input_text = output_text
        return output_text
        time.sleep(0.2)
    

In [71]:
show_prediction("Humans have lived", 20)

'Humans have lived in'

In [72]:
import time
input_text = "Humans have lived"
for i in range(20):
    output_text = prediction(model,dictionary,input_text)
    print(output_text)
    input_text = output_text
    time.sleep(0.2)
    

Humans have lived in
Humans have lived in societies
Humans have lived in societies without
Humans have lived in societies without formal
Humans have lived in societies without formal hierarchies
Humans have lived in societies without formal hierarchies long
Humans have lived in societies without formal hierarchies long before
Humans have lived in societies without formal hierarchies long before the
Humans have lived in societies without formal hierarchies long before the establishment
Humans have lived in societies without formal hierarchies long before the establishment of
Humans have lived in societies without formal hierarchies long before the establishment of states
Humans have lived in societies without formal hierarchies long before the establishment of states realms
Humans have lived in societies without formal hierarchies long before the establishment of states realms or
Humans have lived in societies without formal hierarchies long before the establishment of states realms or 

In [73]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

## Calculating Accuracy

In [74]:
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x) # Get model predictions
            _, predicted = torch.max(outputs, dim=1) # Get the predicted word indices
            correct += (predicted == batch_y).sum().item() # Compare with actual labels
            total += batch_y.size(0)
    accuracy = correct / total * 100
    return accuracy
accuracy = calculate_accuracy(model, dataloader, device) # Compute accuracy
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 97.19%
